# Building a Full Transformer (LLM) from Scratch in JAX

This notebook implements a complete **GPT-style decoder-only Transformer** in pure JAX,
following the architecture used by modern Large Language Models (GPT, LLaMA, etc.).

## Architecture Overview

```
Input tokens  [B, T]  (batch of integer token IDs)
      │
      ▼
Token Embedding  +  Positional Embedding
      │
      ▼  (repeated N times)
┌─────────────────────────────────────────┐
│         Transformer Block               │
│                                         │
│  RMSNorm                                │
│     ↓                                   │
│  Causal Self-Attention  (multi-head)    │
│     ↓                                   │
│  Residual Add  ( x = x + attn(x) )     │
│                                         │
│  RMSNorm                                │
│     ↓                                   │
│  Feed-Forward Network  (SwiGLU)         │
│     ↓                                   │
│  Residual Add  ( x = x + ffn(x) )      │
└─────────────────────────────────────────┘
      │
      ▼
Final RMSNorm
      │
      ▼
Linear projection → logits  [B, T, vocab_size]
```

## What you will learn
1. JAX's functional programming style (pure functions, explicit params)
2. RMSNorm — the normalisation used in LLaMA/Gemma
3. Causal self-attention with masking
4. Rotary Position Embedding (RoPE)
5. SwiGLU feed-forward network
6. Full Transformer forward pass
7. Cross-entropy loss and training loop with Optax
8. Text generation (greedy and temperature sampling)

## Key JAX Concepts Used
- `jax.random.PRNGKey` — explicit random state
- `jax.jit` — XLA compilation for speed
- `jax.value_and_grad` — compute loss + gradients in one pass
- `jax.tree_util.tree_map` — apply functions to nested param dicts
- `jnp` — JAX's NumPy-compatible array API

## Step 1 — Imports and Configuration

We import JAX and its NumPy interface (`jnp`), plus Optax for the AdamW optimiser.
In JAX, all random operations require an explicit **PRNG key** — there is no global
random state. This makes code reproducible and JIT-compilable.

In [2]:
import jax                          # core JAX library
import jax.numpy as jnp            # JAX's NumPy — same API, runs on GPU/TPU
import numpy as np                  # standard NumPy for data preparation
import optax                        # gradient-based optimisers for JAX
from functools import partial       # partial function application
from typing import NamedTuple       # typed parameter containers

# Print JAX version and available devices
print(f"JAX version  : {jax.__version__}")
print(f"JAX devices  : {jax.devices()}")  # will show GPU/TPU if available

JAX version  : 0.10.2
JAX devices  : [CpuDevice(id=0)]


## Step 2 — Hyperparameters

All model hyperparameters are collected in a single config dictionary.
This is a deliberate design choice in JAX: make all configuration explicit
and pass it around rather than storing it inside objects.

| Parameter | Meaning |
|-----------|--------|
| `vocab_size` | Number of unique tokens in the vocabulary |
| `d_model` | Embedding dimension — width of every hidden layer |
| `n_heads` | Number of attention heads (d_model must be divisible by n_heads) |
| `d_head` | Dimension per attention head = d_model / n_heads |
| `d_ff` | Inner dimension of the feed-forward network (typically 8/3 × d_model for SwiGLU) |
| `n_layers` | Number of stacked Transformer blocks |
| `max_seq_len` | Maximum sequence length the model can process |
| `dropout_rate` | Fraction of neurons randomly zeroed during training |

In [5]:
# ── Model hyperparameters ──────────────────────────────────────────────────
cfg = {
    "vocab_size"   : 1000,    # vocabulary size (tokens the model can predict)
    "d_model"      : 128,     # embedding / hidden dimension
    "n_heads"      : 4,       # number of parallel attention heads
    "d_head"       : 32,      # dimension per head  = d_model / n_heads
    "d_ff"         : 341,     # SwiGLU inner dim ≈ (8/3) * d_model, rounded
    "n_layers"     : 4,       # number of Transformer blocks stacked
    "max_seq_len"  : 64,      # maximum context length in tokens
    "dropout_rate" : 0.1,     # dropout probability (set to 0.0 at inference)
}

# Sanity check: d_model must be divisible by n_heads
assert cfg["d_model"] % cfg["n_heads"] == 0, \
    f"d_model ({cfg['d_model']}) must be divisible by n_heads ({cfg['n_heads']})"

print("Config validated ✓")
print(f"Parameters (approx): "
      f"{cfg['vocab_size']*cfg['d_model'] + cfg['n_layers']*(cfg['d_model']**2 * 4):,}")

Config validated ✓
Parameters (approx): 390,144


## Step 3 — Parameter Initialisation

In JAX we never use an object like `nn.Linear`. Instead, a linear layer is just a
**weight matrix** (a plain JAX array). All parameters live in a nested Python dictionary
called `params`. This is the *functional programming* style.

### Initialisation strategy
- **Embeddings:** small random normal (std = 0.02) — standard for GPT-style models
- **Attention Q/K/V/O projections:** normal with std = 0.02
- **FFN weights:** same, with the output projection scaled by 1/√(2×n_layers)
  to keep residual stream variance stable at initialisation
- **RMSNorm scale (γ):** ones — so the first forward pass is identity normalisation
- **Biases:** none — modern LLMs typically omit biases for simplicity

In [ ]:
def init_params(cfg: dict, key: jax.random.PRNGKey) -> dict:
    """
    Initialise all model parameters.

    Args:
        cfg : hyperparameter dictionary
        key : JAX PRNG key for reproducible random initialisation

    Returns:
        params : nested dict  { 'embed': ..., 'layers': [...], 'norm': ..., 'head': ... }
    """
    # Helper: split the key and sample from a normal distribution
    def normal(key, shape, std=0.02):
        # jax.random.normal returns values from N(0,1); we scale by std
        return jax.random.normal(key, shape) * std

    V  = cfg["vocab_size"]    # shorthand
    D  = cfg["d_model"]
    H  = cfg["n_heads"]
    Dh = cfg["d_head"]        # = D // H
    Df = cfg["d_ff"]
    L  = cfg["n_layers"]

    # ── Token embedding table  [vocab_size, d_model] ──────────────────────
    # Each row is the learned vector representation of one vocabulary token
    key, k = jax.random.split(key)
    tok_emb = normal(k, (V, D))          # shape: (vocab_size, d_model)

    # ── Positional embedding table  [max_seq_len, d_model] ────────────────
    # Each row encodes the position of a token in the sequence
    # (Later we also add RoPE inside attention, but learned pos-emb is common)
    key, k = jax.random.split(key)
    pos_emb = normal(k, (cfg["max_seq_len"], D))  # shape: (max_seq_len, d_model)

    # ── Transformer blocks ─────────────────────────────────────────────────
    layers = []                          # list of length n_layers
    for _ in range(L):
        # RMSNorm scale vectors (γ): initialised to ones → identity at start
        key, k1, k2 = jax.random.split(key, 3)
        norm1_w = jnp.ones(D)            # scale for pre-attention norm
        norm2_w = jnp.ones(D)            # scale for pre-FFN norm

        # ── Attention weight matrices ─────────────────────────────────────
        # Fused QKV: project d_model → 3 * d_model in one matrix multiply
        key, k = jax.random.split(key)
        W_qkv = normal(k, (D, 3 * D))   # shape: (d_model, 3*d_model)

        # Output projection: concatenated heads → d_model
        key, k = jax.random.split(key)
        W_o   = normal(k, (D, D))        # shape: (d_model, d_model)

        # ── SwiGLU FFN weight matrices ────────────────────────────────────
        # SwiGLU uses THREE weight matrices: W1 (gate), W2 (value), W3 (down)
        # Output = (SiLU(x @ W1) ⊙ (x @ W2)) @ W3
        key, k1, k2, k3 = jax.random.split(key, 4)
        W1 = normal(k1, (D, Df))         # gate projection:   d_model → d_ff
        W2 = normal(k2, (D, Df))         # value projection:  d_model → d_ff
        # Output projection scaled by 1/√(2L) to keep residual variance stable
        W3 = normal(k3, (Df, D), std=0.02 / (2 * L) ** 0.5)  # d_ff → d_model

        layers.append({
            "norm1"  : norm1_w,   # RMSNorm γ before self-attention
            "norm2"  : norm2_w,   # RMSNorm γ before FFN
            "W_qkv"  : W_qkv,    # fused query-key-value projection
            "W_o"    : W_o,       # attention output projection
            "W1"     : W1,        # SwiGLU gate weight
            "W2"     : W2,        # SwiGLU value weight
            "W3"     : W3,        # SwiGLU down-projection weight
        })

    # ── Final RMSNorm before the language model head ───────────────────────
    final_norm_w = jnp.ones(D)           # shape: (d_model,)

    # ── Language model head: d_model → vocab_size (no bias) ───────────────
    # Projects the final hidden state to per-token vocabulary logits
    key, k = jax.random.split(key)
    lm_head = normal(k, (D, V))          # shape: (d_model, vocab_size)

    return {
        "tok_emb"    : tok_emb,      # token embedding table
        "pos_emb"    : pos_emb,      # positional embedding table
        "layers"     : layers,       # list of transformer block param dicts
        "final_norm" : final_norm_w, # final RMSNorm scale
        "lm_head"    : lm_head,      # output projection to vocabulary
    }


# Initialise parameters with a fixed seed for reproducibility
key    = jax.random.PRNGKey(42)    # fixed seed: 42 → deterministic init
params = init_params(cfg, key)     # all weights in one nested dict

# Count total number of scalar parameters
total = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total parameters: {total:,}")
print(f"Parameter keys  : {list(params.keys())}")

## Step 4 — RMSNorm (Root Mean Square Normalisation)

RMSNorm (Zhang & Sennrich, 2019) is a simpler alternative to LayerNorm used in
LLaMA, Gemma, and most modern LLMs. It normalises by the **root mean square**
of the activations rather than their mean and variance.

$$
\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma
\qquad\text{where}\qquad
\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \varepsilon}
$$

**Why RMSNorm?**
- No mean subtraction → slightly fewer operations than LayerNorm
- Empirically matches or exceeds LayerNorm quality at LLM scale
- Pre-norm placement (before each sub-layer) stabilises deep network training

$\gamma$ (gamma) is a learnable scale vector of shape `(d_model,)`, initialised to ones.

In [ ]:
def rms_norm(x: jnp.ndarray, w: jnp.ndarray, eps: float = 1e-6) -> jnp.ndarray:
    """
    Root Mean Square Layer Normalisation.

    Args:
        x   : input tensor of shape (..., d_model)
        w   : learnable scale γ of shape (d_model,)
        eps : small constant for numerical stability (avoids division by zero)

    Returns:
        Normalised tensor of the same shape as x
    """
    # Compute mean of squared values along the last dimension (the feature dim)
    # keepdims=True preserves the dimension for broadcasting
    ms = jnp.mean(x ** 2, axis=-1, keepdims=True)  # shape: (..., 1)

    # Compute the RMS: sqrt of mean-square + epsilon
    rms = jnp.sqrt(ms + eps)                         # shape: (..., 1)

    # Normalise x by its RMS, then scale by learnable weight γ
    # w has shape (d_model,) — broadcast over batch and sequence dims
    return (x / rms) * w                             # shape: (..., d_model)


# ── Quick sanity check ──────────────────────────────────────────────────────
test_x = jnp.array([[1.0, 2.0, 3.0, 4.0],
                     [5.0, 6.0, 7.0, 8.0]])          # shape: (2, 4)
test_w = jnp.ones(4)                                  # γ = ones → scale = 1
normed = rms_norm(test_x, test_w)

print("Input  :", test_x)
print("Normed :", normed)
# RMS of normed output should be ≈ 1 for each row
print("RMS of output rows:", jnp.sqrt(jnp.mean(normed**2, axis=-1)))

## Step 5 — Rotary Position Embedding (RoPE)

RoPE (Su et al., 2021) encodes **relative** position information by rotating
query and key vectors before computing attention scores. Used in LLaMA, Gemma,
Mistral, and most modern LLMs.

### How it works

Each pair of dimensions $(q_{2i}, q_{2i+1})$ is rotated by angle $m \cdot \theta_i$
where $m$ is the token position and $\theta_i = 10000^{-2i/d}$:

$$
\begin{bmatrix} q_{2i}' \\ q_{2i+1}' \end{bmatrix}
=
\begin{bmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{bmatrix}
\begin{bmatrix} q_{2i} \\ q_{2i+1} \end{bmatrix}
$$

After rotation, the dot product $q_m^T k_n$ depends only on the **relative position**
$m - n$, giving the model distance-awareness without a separate embedding table.

**Key advantage over learned positional embeddings:**
RoPE generalises to sequence lengths beyond what was seen during training.

In [ ]:
def precompute_rope_freqs(d_head: int, max_seq_len: int,
                           theta: float = 10000.0) -> jnp.ndarray:
    """
    Precompute RoPE rotation angles for all positions and head dimensions.

    Args:
        d_head      : dimension of each attention head
        max_seq_len : maximum sequence length
        theta       : base frequency (10000 is standard, 500000 for long context)

    Returns:
        freqs_cis : complex rotation factors, shape (max_seq_len, d_head//2)
    """
    # Compute inverse frequencies: θ_i = 1 / (theta ^ (2i / d_head))
    # Shape: (d_head // 2,)  — one frequency per pair of dimensions
    inv_freq = 1.0 / (
        theta ** (jnp.arange(0, d_head, 2).astype(jnp.float32) / d_head)
    )                                               # shape: (d_head//2,)

    # Position indices: 0, 1, 2, ..., max_seq_len-1
    positions = jnp.arange(max_seq_len).astype(jnp.float32)  # shape: (T,)

    # Outer product: each position × each frequency = rotation angle
    # Shape: (max_seq_len, d_head//2)
    angles = jnp.outer(positions, inv_freq)         # shape: (T, d_head//2)

    # Represent as complex numbers: e^(i*angle) = cos(angle) + i*sin(angle)
    # This allows rotation via complex multiplication
    freqs_cis = jnp.exp(1j * angles)               # shape: (T, d_head//2), complex

    return freqs_cis


def apply_rope(x: jnp.ndarray, freqs_cis: jnp.ndarray) -> jnp.ndarray:
    """
    Apply Rotary Position Embedding to a query or key tensor.

    Args:
        x         : query or key, shape (B, n_heads, T, d_head)
        freqs_cis : precomputed complex frequencies, shape (T, d_head//2)

    Returns:
        x_rope : rotated tensor, same shape as x
    """
    B, H, T, D = x.shape

    # Reshape pairs of dimensions into complex numbers
    # (B, H, T, D) → (B, H, T, D//2) complex
    x_c = x.reshape(B, H, T, D // 2, 2)            # split into pairs
    x_c = x_c[..., 0] + 1j * x_c[..., 1]           # combine as complex

    # Expand freqs_cis to match (1, 1, T, D//2) for broadcasting
    f = freqs_cis[None, None, :T, :]                 # shape: (1, 1, T, D//2)

    # Rotate: multiply complex x by complex rotation factor
    # Real(x*f) = x_real*cos - x_imag*sin  (rotated real part)
    # Imag(x*f) = x_real*sin + x_imag*cos  (rotated imaginary part)
    x_rotated = x_c * f                              # complex multiplication

    # Convert back to real by stacking real and imaginary parts
    x_out = jnp.stack([x_rotated.real, x_rotated.imag], axis=-1)  # (..., D//2, 2)
    x_out = x_out.reshape(B, H, T, D)               # restore original shape

    return x_out.astype(x.dtype)                     # keep original dtype


# Precompute for the whole training run (done once)
rope_freqs = precompute_rope_freqs(cfg["d_head"], cfg["max_seq_len"])
print(f"RoPE freqs shape: {rope_freqs.shape}  (seq_len, d_head//2) complex")

## Step 6 — Causal Multi-Head Self-Attention

Self-attention allows every token to attend to every previous token (causal = left-to-right only).

### Formula

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M\right) V
$$

Where:
- $Q, K, V$ are query, key, value projections of the input
- $\sqrt{d_k}$ is the scaling factor that prevents dot products from growing too large
- $M$ is the **causal mask**: $M_{ij} = 0$ if $j \le i$ (token $i$ can attend to $j$), else $-\infty$

### Multi-head
Run $H$ independent attention operations in parallel, each in a $(d_{\text{model}}/H)$-dimensional subspace.
Concatenate outputs and project back to $d_{\text{model}}$.

### Implementation details
- Fused QKV: project input → $[Q, K, V]$ in a single matrix multiply
- Causal mask: lower-triangular boolean mask, broadcast over batch and heads
- RoPE: applied to Q and K after projection, before score computation

In [ ]:
def causal_self_attention(
    x: jnp.ndarray,              # input tensor, shape (B, T, D)
    W_qkv: jnp.ndarray,          # fused QKV weight, shape (D, 3*D)
    W_o: jnp.ndarray,            # output projection, shape (D, D)
    rope_freqs: jnp.ndarray,     # precomputed RoPE freqs, shape (T, d_head//2)
    n_heads: int,                 # number of attention heads
) -> jnp.ndarray:
    """
    Causal multi-head self-attention with RoPE.

    Returns tensor of same shape as x: (B, T, D)
    """
    B, T, D = x.shape              # batch size, sequence length, d_model
    d_head  = D // n_heads         # dimension per attention head

    # ── Step 1: Fused QKV projection ──────────────────────────────────────
    # Project input to queries, keys, and values in one matrix multiply.
    # x @ W_qkv: (B, T, D) @ (D, 3D) → (B, T, 3D)
    qkv = x @ W_qkv                                  # shape: (B, T, 3*D)

    # Split along the last dimension into Q, K, V
    # Each has shape (B, T, D)
    Q, K, V = jnp.split(qkv, 3, axis=-1)            # three tensors (B, T, D)

    # ── Step 2: Reshape to per-head tensors ───────────────────────────────
    # Reshape (B, T, D) → (B, T, n_heads, d_head) then transpose to (B, n_heads, T, d_head)
    # This lets each head attend independently in its own subspace
    def split_heads(t):
        return t.reshape(B, T, n_heads, d_head).transpose(0, 2, 1, 3)

    Q = split_heads(Q)             # shape: (B, n_heads, T, d_head)
    K = split_heads(K)             # shape: (B, n_heads, T, d_head)
    V = split_heads(V)             # shape: (B, n_heads, T, d_head)

    # ── Step 3: Apply Rotary Position Embedding to Q and K ────────────────
    # RoPE encodes position by rotating Q and K vectors.
    # After rotation, dot(Q_m, K_n) depends on relative position m-n.
    Q = apply_rope(Q, rope_freqs)  # shape unchanged: (B, n_heads, T, d_head)
    K = apply_rope(K, rope_freqs)  # shape unchanged: (B, n_heads, T, d_head)

    # ── Step 4: Scaled dot-product attention scores ────────────────────────
    # scores[b, h, i, j] = how much token i attends to token j
    # K.transpose(-2,-1): (B, n_heads, d_head, T)
    scale  = d_head ** -0.5        # 1/sqrt(d_head) — prevents large dot products
    scores = Q @ K.transpose(0, 1, 3, 2) * scale  # shape: (B, n_heads, T, T)

    # ── Step 5: Causal mask (lower-triangular) ─────────────────────────────
    # Token at position i must NOT attend to positions j > i (future tokens).
    # We set those positions to -inf so softmax gives them weight ≈ 0.
    # jnp.tril creates a lower-triangular matrix of True values
    mask = jnp.tril(jnp.ones((T, T), dtype=bool))   # shape: (T, T)
    # Broadcast mask over batch and head dimensions: (1, 1, T, T)
    # where mask is False (future positions), set score to -1e9 (≈ -inf)
    scores = jnp.where(mask[None, None, :, :], scores, -1e9)  # shape: (B, n_heads, T, T)

    # ── Step 6: Softmax to get attention weights ───────────────────────────
    # Apply softmax over the last dimension (the key/value sequence dimension)
    # Each row sums to 1: weights[b, h, i, :] is a probability distribution
    attn_w = jax.nn.softmax(scores, axis=-1)         # shape: (B, n_heads, T, T)

    # ── Step 7: Weighted sum of values ────────────────────────────────────
    # Each query position gets a weighted combination of value vectors
    # attn_w @ V: (B, n_heads, T, T) @ (B, n_heads, T, d_head) → (B, n_heads, T, d_head)
    out = attn_w @ V                                  # shape: (B, n_heads, T, d_head)

    # ── Step 8: Merge heads and project back to d_model ───────────────────
    # Transpose: (B, n_heads, T, d_head) → (B, T, n_heads, d_head)
    # Reshape:   (B, T, n_heads * d_head) = (B, T, D)
    out = out.transpose(0, 2, 1, 3).reshape(B, T, D)  # shape: (B, T, D)

    # Final linear projection: mix information across heads
    # out @ W_o: (B, T, D) @ (D, D) → (B, T, D)
    return out @ W_o                                  # shape: (B, T, D)


print("causal_self_attention defined ✓")

## Step 7 — SwiGLU Feed-Forward Network

The Feed-Forward Network (FFN) processes each token position independently.
Modern LLMs use **SwiGLU** (Shazeer 2020) instead of a standard ReLU FFN.

### Standard FFN (BERT/GPT-2 style)
$$\text{FFN}(x) = \text{GELU}(x W_1) W_2$$

### SwiGLU (LLaMA style)
$$\text{SwiGLU}(x) = \big(\text{SiLU}(x W_1) \odot (x W_2)\big) W_3$$

Where:
- $W_1$: gate projection — passed through SiLU activation
- $W_2$: value projection — used as an element-wise gate
- $W_3$: down-projection — maps back to `d_model`
- $\odot$: element-wise (Hadamard) product
- $\text{SiLU}(x) = x \cdot \sigma(x)$ — smooth, non-saturating activation

**Why three matrices instead of two?** The gating mechanism allows the network to
*selectively* pass information based on its content. Empirically, SwiGLU produces
better perplexity than GELU FFNs with the same parameter count.

In [ ]:
def swiglu_ffn(
    x: jnp.ndarray,    # input, shape (B, T, D)
    W1: jnp.ndarray,   # gate weight,  shape (D, d_ff)
    W2: jnp.ndarray,   # value weight, shape (D, d_ff)
    W3: jnp.ndarray,   # down weight,  shape (d_ff, D)
) -> jnp.ndarray:
    """
    SwiGLU Feed-Forward Network.

    Computes: (SiLU(x @ W1) ⊙ (x @ W2)) @ W3

    Returns tensor of same shape as x: (B, T, D)
    """
    # ── Gate path: project then apply SiLU activation ─────────────────────
    # SiLU(x) = x * sigmoid(x) — smooth activation without saturation for x > 0
    # x @ W1: (B, T, D) @ (D, d_ff) → (B, T, d_ff)
    gate = jax.nn.silu(x @ W1)   # shape: (B, T, d_ff)  — gating signal

    # ── Value path: plain linear projection (no activation) ───────────────
    # x @ W2: (B, T, D) @ (D, d_ff) → (B, T, d_ff)
    val  = x @ W2                 # shape: (B, T, d_ff)  — value signal

    # ── Element-wise gating (Hadamard product) ─────────────────────────────
    # The gate selectively amplifies or suppresses each feature in val
    # This is the "gated linear unit" core idea
    gated = gate * val            # shape: (B, T, d_ff)  — gated features

    # ── Down-projection back to d_model ───────────────────────────────────
    # gated @ W3: (B, T, d_ff) @ (d_ff, D) → (B, T, D)
    return gated @ W3             # shape: (B, T, D)


print("swiglu_ffn defined ✓")

## Step 8 — Transformer Block

A single Transformer block applies:
1. **Pre-norm attention:** `x = x + Attention(RMSNorm(x))`
2. **Pre-norm FFN:** `x = x + FFN(RMSNorm(x))`

The **residual connections** (`x + ...`) are critical:
- They create direct gradient paths back to the input — preventing vanishing gradients
- They allow the model to learn *corrections* to the existing representation rather than complete transformations
- Mathematically: $\partial(x + f(x))/\partial x = 1 + \partial f/\partial x$ — the `+1` ensures gradient magnitude ≥ 1

The **pre-norm** placement (normalise before the sub-layer, not after) is used by
all modern LLMs (LLaMA, Mistral, Gemma) because it makes training more stable
and eliminates the need for careful learning rate warmup.

In [ ]:
def transformer_block(
    x: jnp.ndarray,         # input, shape (B, T, D)
    layer_params: dict,      # dict with norm1, norm2, W_qkv, W_o, W1, W2, W3
    rope_freqs: jnp.ndarray, # precomputed RoPE frequencies
    n_heads: int,            # number of attention heads
) -> jnp.ndarray:
    """
    One full Transformer block: pre-norm attention + pre-norm FFN, both with residuals.

    Returns tensor of same shape as x: (B, T, D)
    """
    # ── Sub-layer 1: Causal Self-Attention ────────────────────────────────

    # Pre-normalisation: normalise x before attention (more stable than post-norm)
    x_normed = rms_norm(x, layer_params["norm1"])  # shape: (B, T, D)

    # Compute causal self-attention on the normalised input
    attn_out = causal_self_attention(
        x_normed,
        layer_params["W_qkv"],   # fused Q/K/V projection weights
        layer_params["W_o"],     # output projection weights
        rope_freqs,              # rotary position frequencies
        n_heads,                 # number of attention heads
    )                                               # shape: (B, T, D)

    # Residual connection: add attention output to the original input
    # This preserves the original information and only adds corrections
    x = x + attn_out                               # shape: (B, T, D)

    # ── Sub-layer 2: SwiGLU Feed-Forward Network ──────────────────────────

    # Pre-normalisation: normalise again before the FFN
    x_normed = rms_norm(x, layer_params["norm2"])  # shape: (B, T, D)

    # Apply SwiGLU FFN to the normalised input
    ffn_out = swiglu_ffn(
        x_normed,
        layer_params["W1"],      # gate projection
        layer_params["W2"],      # value projection
        layer_params["W3"],      # down projection
    )                                               # shape: (B, T, D)

    # Residual connection: add FFN output to current x
    x = x + ffn_out                                # shape: (B, T, D)

    return x                                        # shape: (B, T, D)


print("transformer_block defined ✓")

## Step 9 — Full Transformer Forward Pass

The full model:
1. Looks up token embeddings from the embedding table
2. Adds positional embeddings (position 0, 1, 2, ...)
3. Passes through N Transformer blocks
4. Applies a final RMSNorm
5. Projects to vocabulary logits with the language model head

The output logits can be used to:
- **Train**: compute cross-entropy loss against the next token targets
- **Generate**: sample from the probability distribution over vocabulary

Note: in JAX, the entire forward pass is a **pure function** — it takes `params` and
input tokens, and returns logits. No hidden state, no mutation.

In [ ]:
def forward(
    params: dict,          # all model parameters (from init_params)
    tokens: jnp.ndarray,   # integer token IDs, shape (B, T)
    cfg: dict,             # hyperparameter config
    rope_freqs: jnp.ndarray, # precomputed RoPE frequencies
) -> jnp.ndarray:
    """
    Full Transformer forward pass.

    Args:
        params     : nested dict of all model weights
        tokens     : integer token IDs, shape (B, T)
        cfg        : hyperparameter config dict
        rope_freqs : precomputed RoPE rotation factors

    Returns:
        logits : unnormalised scores over vocabulary, shape (B, T, vocab_size)
    """
    B, T = tokens.shape     # batch size and sequence length

    # ── Step 1: Token embedding lookup ────────────────────────────────────
    # Each integer token ID maps to a learned dense vector
    # params["tok_emb"][i] gives the embedding of token i
    # tok_emb[tokens] performs a parallel lookup for the whole batch
    x = params["tok_emb"][tokens]                   # shape: (B, T, D)

    # ── Step 2: Positional embedding ─────────────────────────────────────
    # Add positional embedding for positions 0, 1, ..., T-1
    # jnp.arange(T) creates [0, 1, 2, ..., T-1]
    # params["pos_emb"][:T] selects the first T positional vectors
    pos = jnp.arange(T)                             # shape: (T,)
    x   = x + params["pos_emb"][pos]                # shape: (B, T, D)
    # Note: pos_emb broadcasts over the batch dimension automatically

    # ── Step 3: Pass through N Transformer blocks ─────────────────────────
    # Each block applies attention + FFN with residual connections
    for i, layer_p in enumerate(params["layers"]):
        x = transformer_block(
            x,            # current hidden state
            layer_p,      # parameters for block i
            rope_freqs,   # shared RoPE frequencies
            cfg["n_heads"]
        )                                           # shape: (B, T, D)  — unchanged

    # ── Step 4: Final RMSNorm ─────────────────────────────────────────────
    # Normalise the hidden states before projection to vocabulary
    # This stabilises training and improves the quality of the output distribution
    x = rms_norm(x, params["final_norm"])           # shape: (B, T, D)

    # ── Step 5: Language model head ───────────────────────────────────────
    # Project from d_model to vocab_size: these are unnormalised log-probabilities
    # (logits). Apply softmax to get probabilities, or cross-entropy for training.
    # x @ params["lm_head"]: (B, T, D) @ (D, vocab_size) → (B, T, vocab_size)
    logits = x @ params["lm_head"]                  # shape: (B, T, vocab_size)

    return logits


# ── Quick shape verification ───────────────────────────────────────────────
dummy_tokens = jnp.ones((2, 16), dtype=jnp.int32)  # batch=2, seq_len=16
logits = forward(params, dummy_tokens, cfg, rope_freqs)
print(f"Input  shape: {dummy_tokens.shape}   (batch, seq_len)")
print(f"Output shape: {logits.shape}  (batch, seq_len, vocab_size)")
print("Forward pass ✓")

## Step 10 — Loss Function

We use **cross-entropy loss** on next-token prediction — the standard training objective
for all autoregressive language models (GPT, LLaMA, etc.).

### Teacher Forcing
During training, the model sees tokens $t_1, t_2, ..., t_{T-1}$ as input and is
trained to predict $t_2, t_3, ..., t_T$ as targets.

```
Input:   [t₁,  t₂,  t₃,  ..., t_{T-1}]  ← tokens 0..T-2
Targets: [t₂,  t₃,  t₄,  ..., t_T]      ← tokens 1..T-1 (shifted left by 1)
```

### Cross-Entropy
$$\mathcal{L} = -\frac{1}{T-1} \sum_{t=1}^{T-1} \log P(t_{t+1} \mid t_1, ..., t_t)$$

In code: `optax.softmax_cross_entropy_with_integer_labels` does softmax + log + negate
in one numerically stable operation.

In [ ]:
def loss_fn(
    params: dict,          # model weights
    tokens: jnp.ndarray,   # integer tokens, shape (B, T)
    cfg: dict,             # hyperparameter config
    rope_freqs: jnp.ndarray, # precomputed RoPE
) -> jnp.ndarray:
    """
    Compute next-token prediction cross-entropy loss.

    Uses teacher forcing: input is tokens[0..T-2], target is tokens[1..T-1].

    Returns:
        scalar mean cross-entropy loss
    """
    # ── Prepare input and target sequences (teacher forcing) ───────────────
    # Input:  all tokens except the last  → tokens[:, :-1], shape (B, T-1)
    # Target: all tokens except the first → tokens[:, 1:],  shape (B, T-1)
    # The model sees token t and must predict token t+1
    inp = tokens[:, :-1]           # shape: (B, T-1)  — model input
    tgt = tokens[:, 1:]            # shape: (B, T-1)  — ground truth next tokens

    # ── Forward pass: get logits for each position ─────────────────────────
    # logits[b, t, :] is the unnormalised distribution over vocabulary
    # for predicting the token at position t+1 given the context up to position t
    logits = forward(params, inp, cfg, rope_freqs)  # shape: (B, T-1, vocab_size)

    # ── Flatten for loss computation ──────────────────────────────────────
    # Cross-entropy expects (N, vocab_size) logits and (N,) integer labels
    # We flatten batch and sequence dimensions together: N = B * (T-1)
    B, Tm1, V = logits.shape       # Tm1 = T - 1
    logits_2d = logits.reshape(B * Tm1, V)  # shape: (B*(T-1), vocab_size)
    tgt_1d    = tgt.reshape(B * Tm1)        # shape: (B*(T-1),)

    # ── Compute per-token cross-entropy ───────────────────────────────────
    # optax.softmax_cross_entropy_with_integer_labels:
    #   1. Applies softmax to logits (numerically stable log-softmax)
    #   2. Looks up the log-probability of the true token
    #   3. Returns negative log-probability (cross-entropy) per token
    per_token_loss = optax.softmax_cross_entropy_with_integer_labels(
        logits_2d, tgt_1d
    )                              # shape: (B*(T-1),)  — one loss value per position

    # Return mean loss over all tokens and all batch examples
    return jnp.mean(per_token_loss)  # scalar


# Verify loss on random data
dummy_tokens = jax.random.randint(jax.random.PRNGKey(1), (2, 16), 0, cfg["vocab_size"])
L = loss_fn(params, dummy_tokens, cfg, rope_freqs)
print(f"Initial loss: {L:.4f}")
# For random weights and vocab_size=1000: expect ~log(1000) ≈ 6.9
print(f"Expected (random): ~{jnp.log(cfg['vocab_size']):.2f}")

## Step 11 — Training Loop

The JAX training loop follows the functional pattern:

```
for each batch:
    loss, grads = value_and_grad(loss_fn)(params, batch)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
```

Key JAX idioms:
- `jax.value_and_grad`: computes both the loss value and gradients in one pass (efficient)
- `@jax.jit`: compiles the step function with XLA — huge speedup on GPU/TPU
- `optax.apply_updates`: `params + updates` using `jax.tree_util.tree_map`
- No `.backward()`, no `optimizer.step()` — all explicit and functional

### Optimiser: AdamW with Cosine LR Schedule
- **AdamW**: Adam with decoupled weight decay — standard for LLM training
- **Cosine decay**: learning rate starts at peak, smoothly decays to near zero
- **Warmup**: linearly ramp lr from 0 → peak over first N steps (prevents instability)

In [ ]:
# ── Optimiser: AdamW with linear warmup + cosine decay ────────────────────
NUM_STEPS    = 200       # total training steps (small for demonstration)
WARMUP_STEPS = 20        # steps to linearly ramp learning rate from 0 → peak
PEAK_LR      = 3e-4      # peak learning rate (standard for small LLMs)
WEIGHT_DECAY = 0.1       # L2 regularisation coefficient

# Cosine decay schedule with warmup
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value   = 0.0,         # start from lr=0 (warmup begins)
    peak_value   = PEAK_LR,     # max learning rate reached after warmup
    warmup_steps = WARMUP_STEPS,# number of steps to warm up
    decay_steps  = NUM_STEPS,   # total steps (lr decays to end_value)
    end_value    = PEAK_LR * 0.1  # final lr = 10% of peak
)

# Chain: gradient clipping → AdamW with lr schedule
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),          # clip gradient norm to 1.0 (prevents explosion)
    optax.adamw(lr_schedule, weight_decay=WEIGHT_DECAY)  # AdamW optimiser
)

# Initialise optimiser state (Adam momentum vectors m, v + step counter)
# opt_state has the same tree structure as params
opt_state = optimizer.init(params)


# ── JIT-compiled training step ─────────────────────────────────────────────
@jax.jit
def train_step(
    params: dict,
    opt_state,
    tokens: jnp.ndarray,
):
    """
    One gradient update step.

    Returns:
        new_params  : updated model parameters
        new_opt_state: updated optimiser state (Adam m, v vectors)
        loss        : scalar loss value for logging
    """
    # jax.value_and_grad:
    #   Computes f(x) AND ∂f/∂x in a single backward pass
    #   argnums=0 → differentiate w.r.t. the first argument (params)
    loss, grads = jax.value_and_grad(loss_fn)(params, tokens, cfg, rope_freqs)

    # Compute parameter updates from gradients using the optimiser
    # optimizer.update returns: (updates, new_opt_state)
    # updates has the same tree structure as grads and params
    updates, new_opt_state = optimizer.update(grads, opt_state, params)

    # Apply updates: new_params = params + updates  (element-wise for each leaf)
    # optax.apply_updates uses jax.tree_util.tree_map under the hood
    new_params = optax.apply_updates(params, updates)

    return new_params, new_opt_state, loss


# ── Synthetic data: random token sequences ────────────────────────────────
# In a real LLM, this would be batches of tokenised text from a large corpus
def get_batch(key, batch_size=4, seq_len=32):
    """Sample a random batch of token sequences."""
    return jax.random.randint(
        key,
        shape=(batch_size, seq_len),
        minval=0,
        maxval=cfg["vocab_size"]
    )


# ── Training loop ─────────────────────────────────────────────────────────
print(f"Training for {NUM_STEPS} steps...")
print(f"{'Step':>6}  {'Loss':>8}  {'LR':>10}")
print("-" * 30)

data_key = jax.random.PRNGKey(99)   # separate key for data generation

for step in range(NUM_STEPS):
    # Generate a new random batch each step
    data_key, k = jax.random.split(data_key)   # always split before use
    batch = get_batch(k)

    # One gradient update step (compiled after first call)
    params, opt_state, loss = train_step(params, opt_state, batch)

    # Log every 40 steps
    if step % 40 == 0 or step == NUM_STEPS - 1:
        current_lr = lr_schedule(step)          # current learning rate
        print(f"{step:6d}  {loss.item():8.4f}  {current_lr:.2e}")

print("\nTraining complete ✓")

## Step 12 — Text Generation

At inference time, we generate text autoregressively:
1. Start with a prompt (sequence of token IDs)
2. Run the forward pass to get logits for the next token
3. Sample from (or take argmax of) the next-token distribution
4. Append the sampled token to the sequence
5. Repeat until we reach `max_new_tokens`

### Sampling Strategies

| Strategy | How | When to use |
|----------|-----|-------------|
| **Greedy** | Always pick argmax | Deterministic, fast, repetitive |
| **Temperature** | Divide logits by T before softmax | T<1: sharper, T>1: more random |
| **Top-k** | Keep only top-k logits, zero rest | Avoid very unlikely tokens |
| **Top-p (nucleus)** | Keep smallest set with cumulative prob ≥ p | Adaptive, commonly used |

Here we implement **temperature sampling** for simplicity.

In [ ]:
def generate(
    params: dict,              # trained model parameters
    prompt_tokens: jnp.ndarray, # starting token IDs, shape (1, T_prompt)
    max_new_tokens: int,       # number of new tokens to generate
    cfg: dict,                 # model configuration
    rope_freqs: jnp.ndarray,   # RoPE frequencies
    temperature: float = 1.0,  # sampling temperature (1.0 = standard)
    key: jax.random.PRNGKey = None,  # PRNG key for sampling (None = greedy)
) -> jnp.ndarray:
    """
    Autoregressive text generation.

    Args:
        temperature: float
            - temperature > 1.0: softer distribution (more random)
            - temperature < 1.0: sharper distribution (more deterministic)
            - temperature = 0.0 (or key=None): greedy decoding (argmax)

    Returns:
        full_sequence: integer tokens including prompt, shape (1, T_prompt + max_new_tokens)
    """
    # Start with the prompt tokens; we'll append to this
    seq = prompt_tokens              # shape: (1, T_prompt)

    for step in range(max_new_tokens):
        # Truncate if sequence exceeds max_seq_len
        # Only look at the last max_seq_len tokens (sliding window context)
        context = seq[:, -cfg["max_seq_len"]:]  # shape: (1, min(T, max_seq_len))

        # Forward pass: get logits for all positions
        # We only need the logits for the LAST position (next token prediction)
        logits = forward(params, context, cfg, rope_freqs)  # (1, T, vocab)

        # Extract logits for the last token position only
        next_logits = logits[:, -1, :]   # shape: (1, vocab_size)

        if key is None or temperature == 0.0:
            # ── Greedy decoding: always pick the most likely token ─────────
            next_token = jnp.argmax(next_logits, axis=-1, keepdims=True)  # (1, 1)
        else:
            # ── Temperature sampling ──────────────────────────────────────
            # Divide logits by temperature before softmax:
            # - temperature < 1: concentrates probability on top tokens (safer)
            # - temperature > 1: spreads probability more evenly (more creative)
            scaled_logits = next_logits / temperature  # shape: (1, vocab_size)

            # Convert logits to probabilities
            probs = jax.nn.softmax(scaled_logits, axis=-1)  # shape: (1, vocab_size)

            # Sample from the categorical distribution
            # jax.random.categorical returns one sample per row
            key, subkey = jax.random.split(key)   # split key before each use
            next_token = jax.random.categorical(
                subkey, jnp.log(probs + 1e-10), axis=-1
            )[:, None]                            # shape: (1, 1)

        # Append the new token to the growing sequence
        seq = jnp.concatenate([seq, next_token], axis=1)  # (1, T+1)

    return seq  # shape: (1, T_prompt + max_new_tokens)


# ── Run generation ─────────────────────────────────────────────────────────
# Start with a short prompt (token IDs 1, 2, 3)
prompt = jnp.array([[1, 2, 3]], dtype=jnp.int32)  # shape: (1, 3)

# Greedy generation (deterministic)
greedy_out = generate(params, prompt, max_new_tokens=10, cfg=cfg,
                      rope_freqs=rope_freqs, key=None)
print("Greedy output (token IDs):", greedy_out[0].tolist())

# Temperature sampling (random)
gen_key = jax.random.PRNGKey(7)
sampled_out = generate(params, prompt, max_new_tokens=10, cfg=cfg,
                       rope_freqs=rope_freqs, temperature=0.8, key=gen_key)
print("Sampled output (token IDs):", sampled_out[0].tolist())
print("\nGeneration ✓")

## Summary

We have built a complete GPT-style Transformer for language modelling in pure JAX.

### What we implemented

| Component | Function | Key concept |
|-----------|----------|-------------|
| `init_params` | Initialise all weights | Explicit param dict; no objects |
| `rms_norm` | Root Mean Square Norm | Simpler than LayerNorm; used in LLaMA |
| `precompute_rope_freqs` | Precompute RoPE angles | Relative position via rotation |
| `apply_rope` | Apply RoPE to Q, K | Complex multiplication = rotation |
| `causal_self_attention` | Multi-head attention | Lower-triangular mask; scale by 1/√d |
| `swiglu_ffn` | SwiGLU Feed-Forward | Gated: SiLU(xW₁) ⊙ xW₂, then xW₃ |
| `transformer_block` | One full block | Pre-norm + residual for both sub-layers |
| `forward` | Full forward pass | Embed → N blocks → norm → project |
| `loss_fn` | Cross-entropy loss | Next-token prediction with teacher forcing |
| `train_step` | Gradient update | `value_and_grad` + AdamW + `apply_updates` |
| `generate` | Text generation | Greedy or temperature sampling |

### JAX patterns used
- **Functional style:** parameters are explicit dicts, not object attributes
- **`jax.jit`:** XLA-compile the training step for GPU/TPU speed
- **`jax.value_and_grad`:** compute loss + gradients in one efficient backward pass
- **`jax.tree_util`:** operate on nested dicts of arrays (params, grads, updates)
- **Explicit PRNG:** `jax.random.PRNGKey` + `split` — no global random state

### Next steps
- Add **Grouped Query Attention (GQA)** to reduce KV cache memory
- Replace learned positional embedding with **pure RoPE** (remove `pos_emb`)
- Add **KV cache** for efficient autoregressive inference
- Scale up with **`jax.pmap`** or **`jax.sharding`** for multi-GPU training
- Add **real tokenisation** (BPE/SentencePiece) and a real text corpus